# Matched reconstruction of twelve-carrier compact KV, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Keep the candidate frozen. Do **not** regenerate twelve-carrier A. Do **not** overwrite `results_stage_plan4_timing`, hold, or confirm. Do not train an adapter. Do not add calculations.

**Cell 4** (`--phase parity`) runs the generalized N-carrier assembly on the original nine four-carrier hold bases, with their original prompts. It checks selected IDs, slot order, RoPE positions, masks, and continuation logits against the established four-carrier path. If `all_passed` is false, stop.

**Cell 5–6** use the existing nine twelve-carrier sheets and frozen A outputs:
1. **full_kv** — rebuilt completed sender cache
2. **text_result** — RESULT lines as text
3. **generic_r** — generic support plus transferred result states
4. **generic_fresh_r** — the identical support cache, then freshly process the same result tokens at the same packed positions
5. **reconstruct** — freshly process the entire compact sequence (prefix, instruction, separators, result lines)

If condition 4 recovers, changing the result states is enough. If only condition 5 recovers, the support cache or its combination with R also matters. If both fail while text works, inspect framing and presentation before cache adaptation.

The timing summary previously printed zero directional errors because the twelve-carrier scorer omitted those fields. Saved predictions are 44 ineligible-marked-eligible and 1 eligible-marked-ineligible for generic+R. That reporting issue does not explain the accuracy failure.

Upload updated `src/`, `data/plan4_hold.jsonl`, `data/plan4_timing12.jsonl`, `results_stage_plan4_hold/a_artifacts.jsonl`, and `results_stage_plan4_timing/a_artifacts_12.jsonl`.

Download `/kaggle/working/results_stage_plan4_recon12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_recon12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_recon12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_recon12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_recon12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("hold A", (REPO / "results_stage_plan4_hold" / "a_artifacts.jsonl").is_file())
print("timing12 A", (REPO / "results_stage_plan4_timing" / "a_artifacts_12.jsonl").is_file())
print("do not overwrite", WORK / "results_stage_plan4_timing")

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_recon12")
print("Parity first. Then check. Then compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_recon12", "--split", "plan4_timing12", "--phase", "parity"])
print("plan4 recon12 parity exit", rc)
path = WORK / "results_stage_plan4_recon12" / "parity_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_recon12" / "run" / "parity_summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps({"n": data.get("n"), "all_passed": data.get("all_passed")}, indent=2))

In [ ]:
src = WORK / "results_stage_plan4_recon12" / "parity_summary.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_recon12" / "run" / "parity_summary.json"
parity = json.loads(src.read_text(encoding="utf-8"))
if not parity.get("all_passed"):
    raise RuntimeError("Four-carrier assembly parity failed. Do not run the twelve-carrier comparison.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_recon12", "--split", "plan4_timing12", "--phase", "check"])
print("plan4 recon12 check exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_recon12" / "check_summary.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_recon12" / "run" / "check_summary.json"
check = json.loads(src.read_text(encoding="utf-8"))
if not check.get("all_passed"):
    raise RuntimeError("Twelve-carrier reconstruction check failed. Do not run compare.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_recon12", "--split", "plan4_timing12", "--phase", "compare"])
print("plan4 recon12 compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_recon12"
if not (src / "summary.json").is_file() and not (src / "parity_summary.json").is_file():
    src = REPO / "results" / "stage_plan4_recon12" / "run"
print("reading", src)
for name in ("parity_summary.json", "check_summary.json", "summary.json"):
    path = src / name
    if not path.is_file():
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    print("\n====", name, "====")
    keep = {k: data.get(k) for k in ("n", "n_rows", "all_passed", "receiver", "carrier_label_errors", "lead", "parity", "check") if k in data or data.get(k) is not None}
    if "receiver" in keep and isinstance(keep["receiver"], dict):
        keep["receiver"] = {
            cond: {kk: (keep["receiver"].get(cond) or {}).get(kk) for kk in (
                "n", "parse_ok", "truncated", "plan_all_status_correct", "correct",
                "plan_ineligible_marked_eligible", "plan_eligible_marked_ineligible",
                "mean_n_status_correct", "mean_n_kept",
            )}
            for cond in keep["receiver"]
        }
    print(json.dumps(keep, indent=2)[:24000])